# Goal

Попытка "продавить" агента научиться проходить 5-ый уровень. Используем чемпионов из `18n_study_34.3b`. Учитываем опыт `18n_study_19.6{a,b,c,d}` - используем несколько РАМ-ов для 5-го уровня, чтобы не задавить стохастичность, и при этом НЕ используем сложные трейн РАМ-ы, чтобы агент не познавал безнадёгу. Также используем зажатый `prediction_loss`, чтобы стимулировать exploration.

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07.ipynb'

# GRID_SEARCH_SPACE

In [2]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [3]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    HP.general.is_torch_amp_on_rollout = False
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.vision_head.parent = dict(model='18d_world_model_09:183', weights='18d_world_model_09:183')
    HP.vision_head.is_trainable = False

    parent = optuna_trial.suggest_categorical('parent', [
        '18n_ppo_tr_frostbite_07:759',
        '18n_ppo_tr_frostbite_07:761',
        '18n_ppo_tr_frostbite_07:764',
    ])
    
    HP.encoder.parent = dict(model='18d_world_model_09:183', weights=parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = parent 
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = None
    HP.test.env_ram_patches = None
    HP.test.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none', # 0
        
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none1', # 1
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none2', # 2
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none3', # 3
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none4', # 4
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none5', # 5
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none6', # 6
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none7', # 7
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none8', # 8
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none9', # 9
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none10',# 10
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['nine_lives'], 
    ]
    HP.ppo.rollout_env_stories = (
        ['0;0'] +      
        ['0;0'] +      
        ['0;0'] +      
        ['1;0'] +      
        ['2;0'] +      
        ['3;0'] +      
        ['4;0'] +
        ['5;0'] +
        ['6;0'] +
        ['7;0'] +
        ['8;0'] +
        ['9;0'] +
        ['10;0']
    )
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 512 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.05)')
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.1)
    HP.ppo.prediction_loss = dict(coef=0.02) # ЗАНИЖАЕМ КОЭФ, ЧТОБЫ СТИМУЛИРОВАТЬ EXPLORATION ПОВЕДЕНИЕ
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP

# Results

Не, всё глухо, как в танке. Даже ни одного прострела, судя по `episode_lp`.

Ещё одно свидетельство в копилку того, что нужен выдающийся чемпион на прошлом шаге. Надо попробовать вообще прогнать с логикой, похожей на старый `18n_ppo_tr_frostbite_07_old`.

<img src="./img/levels_passed_mean_34.4b.png">
<img src="./img/episode_r_34.4b.png">
<img src="./img/episode_lp_34.4b.png">

**Выводы**
1) попробовать вырастить выдающегося чемпиона на новом сквозном прогоне, ипользуая логику из `18n_ppo_tr_frostbite_07_old` (когда идём от обзов на оптимизации)

# System

In [4]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [5]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_34.4b/18n_study_34.4b.ipynb',
 'optuna_study_name': '18n_study_34.4b',
 'optuna_study_serial': '34.4b',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_34.4b/18n_study_34.4b.optuna'}

In [6]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [7]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [8]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='test/levels_passed_mean',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [9]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [10]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 6
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-08 22:49:33,642] A new study created in Journal with name: 18n_study_34.4b


2026.10.08-22:49:33.928110     0.258 >> Model instance registered, version=938
2026.10.08-22:49:33.944128     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch938.ipynb"
2026.10.08-22:49:33.944800     0.007 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:938"; running launches=1
2026.10.08-22:50:04.909083     0.279 >> Model instance registered, version=939
2026.10.08-22:50:04.923134     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch939.ipynb"
2026.10.08-22:50:04.923463     0.006 >> 5.5 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:939"; running launches=2
2026.10.08-22:50:35.892343     0.267 >> Model instance registered, version=940
2026.10.08-22:50:35.907720     0.007 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch940.ipynb"
2026.10.08-22:50:35.908467     0.007 >> 4.5 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07

In [11]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-08 23:41:18,551] Using an existing study with name '18n_study_34.4b' instead of creating a new one.


2026.10.08-23:41:18.556031     5.039 >> Study statistics: 
2026.10.08-23:41:18.559288     0.003 >> 	Number of finished trials: 6
2026.10.08-23:41:18.560458     0.001 >> 	Number of pruned trials: 0
2026.10.08-23:41:18.561414     0.001 >> 	Number of complete trials: 6
2026.10.08-23:41:18.562460     0.001 >> Best trial:
2026.10.08-23:41:18.563875     0.001 >> 	Value: 2.90625
2026.10.08-23:41:18.564387     0.001 >> 	Model version: 942
2026.10.08-23:41:18.569021     0.005 >> 	Params: 
2026.10.08-23:41:18.570417     0.001 >> 		parent: 18n_ppo_tr_frostbite_07:759


In [12]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.08-23:41:18.576320     0.006 >> Top trials:
2026.10.08-23:41:18.577056     0.001 >> 1) 18n_ppo_tr_frostbite_07:942, value=2.90625
2026.10.08-23:41:18.577426     0.000 >> 2) 18n_ppo_tr_frostbite_07:938, value=2.875
2026.10.08-23:41:18.577777     0.000 >> 3) 18n_ppo_tr_frostbite_07:943, value=2.1875


In [13]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(2.2916666666666665), np.float64(0.4710591272747922))

In [14]:
','.join(map(str, values.tolist()))

'2.875,2.0625,2.15625,1.5625,2.90625,2.1875'